# 8 GE Graph：在Fused Attention期间预取同层o_proj

本章固定开启GE图模式、NPU RMSNorm和Add+RMSNorm融合，对比Prefetch关闭、`self_attn.o_proj`预取16 MiB和32 MiB三组。Prefetch在Fused Attention调用前提交，以`v_cache_after_scatter`（V侧Scatter的返回值，Fused Attention启动前的最后一个依赖）作为启动依赖，Scatter完成后CMO与Fused Attention同时启动、并行执行，在FA读KV Cache期间把o_proj权重搬入L2。

## 1 实验时序

```text
主路径：qkv_proj ── QK-Norm/RoPE/Scatter(VEC) ── Fused Attention ── o_proj MatMul
                                              └──── 预取16/32 MiB ────┘
```

GE Graph不需要模型手动创建Stream。`depend=v_cache_after_scatter`引用V侧Scatter的返回值（Fused Attention启动前的最后一个依赖；K侧Scatter在同一主流上先提交，V侧完成时必然已完成），Scatter完成后CMO与Fused Attention同时启动、并行执行；Fused Attention仍使用原始四维`v_cache`，避免改变KV Cache布局。

In [ ]:
import ast
import copy
import json
import os
import subprocess
import sys
import time
from difflib import SequenceMatcher
from pathlib import Path

import yaml

# 必须在导入torch_npu前选择物理NPU；修改后请重启Notebook内核。
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
os.environ.setdefault('QWEN3_8B_DOWNLOAD_BACKEND', 'modelscope')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CANN_RECIPES_LOG_LEVEL'] = 'INFO'

def locate_repo_root():
    """从当前目录或环境变量定位cann-learning-hub仓库。"""
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    candidates += [base / 'cann-learning-hub' for base in candidates]
    for name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(name):
            candidates.append(Path(os.environ[name]))
    candidates.append(Path('/opt/atomgit/cann-learning-hub'))
    for candidate in dict.fromkeys(map(str, candidates)):
        root = Path(candidate)
        if (root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return root
    raise FileNotFoundError('无法定位仓库，请设置CANN_LEARNING_HUB_ROOT。')

REPO_ROOT = locate_repo_root()
TUTORIAL_DIR = REPO_ROOT / 'tutorials/llm_inference/qwen3_8b'
SRC_DIR = TUTORIAL_DIR / 'src'
RECIPE_ROOT = SRC_DIR / 'inference_scripts/recipe_qwen3_8b'
CONFIG_DIR = RECIPE_ROOT / 'models/qwen/config'
MODEL_SOURCE = RECIPE_ROOT / 'models/qwen/models/modeling_qwen.py'
SOURCE_DIR = REPO_ROOT / 'Sources/model_inference_optimization/qwen3_8b'
RUNTIME_YAML_DIR = SOURCE_DIR / 'recipe_yaml'
OUTPUT_ROOT = SOURCE_DIR / 'run_outputs'
RUNTIME_YAML_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_TEMPLATE = CONFIG_DIR / 'qwen3_8b_1tp_ge_graph_fused.yaml'
PREFETCH_32M_TEMPLATE = CONFIG_DIR / 'qwen3_8b_1tp_ge_graph_fused_o_proj_prefetch_32m.yaml'
PREFETCH_16M_TEMPLATE = CONFIG_DIR / 'qwen3_8b_1tp_ge_graph_fused_o_proj_prefetch_16m.yaml'
MODEL_ID = os.environ.get('QWEN3_8B_MODEL_PATH') or 'Qwen/Qwen3-8B'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from inference_scripts.environment_check import require_unified_environment
require_unified_environment(TUTORIAL_DIR / 'requirements.txt')

print('仓库目录 =', REPO_ROOT)
print('模型源码 =', MODEL_SOURCE)
print('NPU =', os.environ['ASCEND_RT_VISIBLE_DEVICES'])

## 2 检查配置和源码接入

三组必须使用相同模型、融合开关和工作负载。16/32 MiB组只允许Prefetch开关与大小变化。源码检查同时确认预取目标是`self.o_proj.weight`，并且调用位于Fused Attention之前、`o_proj` MatMul之前。

In [ ]:
def load_yaml(path):
    return yaml.safe_load(path.read_text(encoding='utf-8'))

configs = {
    'off': load_yaml(BASELINE_TEMPLATE),
    '16m': load_yaml(PREFETCH_16M_TEMPLATE),
    '32m': load_yaml(PREFETCH_32M_TEMPLATE),
}
for name, config in configs.items():
    model = config['model_config']
    custom = model['custom_params']
    assert model['exe_mode'] == 'ge_graph', name
    assert custom['enable_npu_rmsnorm'] is True, name
    assert custom['enable_add_rmsnorm'] is True, name
    assert 'prefetch_target' not in custom, name
assert configs['off']['model_config']['custom_params']['enable_prefetch'] is False
assert configs['16m']['model_config']['custom_params']['prefetch_size'] == 16 * 1024**2
assert configs['32m']['model_config']['custom_params']['prefetch_size'] == 32 * 1024**2

def comparison_focus(config):
    result = copy.deepcopy(config)
    custom = result['model_config']['custom_params']
    custom.pop('enable_prefetch')
    custom.pop('prefetch_size')
    return result
assert comparison_focus(configs['off']) == comparison_focus(configs['16m'])
assert comparison_focus(configs['off']) == comparison_focus(configs['32m'])

source_text = MODEL_SOURCE.read_text(encoding='utf-8')
source_tree = ast.parse(source_text, filename=str(MODEL_SOURCE))
required_fragments = {
    'target_weight': 'weight=self.o_proj.weight',
    'dependency': 'depend=v_cache_after_scatter',
    'scatter_op': 'v_cache_after_scatter = torch_npu.npu_scatter_nd_update_(',
}
missing = [name for name, text in required_fragments.items() if text not in source_text]
if missing:
    raise RuntimeError(f'o_proj Prefetch源码接入不完整：{missing}')
call_pos = source_text.index('weight=self.o_proj.weight')
fused_pos = source_text.index('fa_ops.npu_fused_infer_attention_score_v2', call_pos)
o_proj_pos = source_text.index('self.o_proj(attn_output)', fused_pos)
assert call_pos < fused_pos < o_proj_pos

print('A/B配置一致性检查通过。')
print('依赖位置 = V侧Scatter输出；CMO与Fused Attention并行执行。')
print('调用顺序 = npu_prefetch(o_proj) -> Fused Attention -> o_proj MatMul')
print('16 MiB覆盖完整32 MiB o_proj的50%；32 MiB完整覆盖。')

## 3 运行三组实验

每组启动独立进程，避免模型、图缓存和NPU状态互相污染。第一次Decode会编译GE图；正式比较使用日志解析出的稳态Decode时间和tokens/s。

In [ ]:
from inference_scripts.inference_qwen3_8b import ensure_recipe_log_ok
from inference_scripts.recipe_workflow import (
    collect_recipe_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)

resolved_model_path, model_source = resolve_model_path(str(MODEL_ID), quiet_model_io=False)

def run_case(tag, template_path):
    work_dir = OUTPUT_ROOT / f'benchmark_{tag}'
    runtime_yaml = RUNTIME_YAML_DIR / f'{tag}.yaml'
    stdout_path = work_dir / 'recipe_logs' / f'{tag}_stdout.log'
    stdout_path.parent.mkdir(parents=True, exist_ok=True)
    prepare_runtime_yaml(template_path, runtime_yaml, model_path=resolved_model_path)
    print_yaml_focus(runtime_yaml)
    cmd = ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--mode', 'offline', '--yaml', str(runtime_yaml)]
    print('运行命令 =', ' '.join(cmd), flush=True)
    started = time.perf_counter()
    with stdout_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            cmd, cwd=RECIPE_ROOT, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{tag}运行失败，日志={stdout_path}')
    case_dir = find_recipe_case_dir(RECIPE_ROOT, runtime_yaml)
    if case_dir is None:
        raise RuntimeError(f'没有找到结果目录：{runtime_yaml.stem}')
    recipe_log_path = case_dir / 'log_0.log'
    ensure_recipe_log_ok(recipe_log_path)
    run_info = {
        'stdout_path': str(stdout_path),
        'stderr_path': str(stdout_path),
        'wall_time_s': time.perf_counter() - started,
        'recipe_case_dir': str(case_dir),
        'recipe_log_path': str(recipe_log_path),
    }
    return collect_recipe_metrics(
        RECIPE_ROOT, work_dir, tag, runtime_yaml, run_info,
        model_id=str(MODEL_ID), resolved_model_path=resolved_model_path,
        model_source=model_source, optimization_mode=tag,
        max_new_tokens=64, max_input_tokens=256, enable_thinking=False,
    )

def show(metrics):
    print(json.dumps({
        'mode': metrics.get('recipe_exe_mode'),
        'prefetch': metrics.get('prefetch_status'),
        'tokens_per_second': metrics.get('tokens_per_second'),
        'decode_time_s': metrics.get('decode_time_s'),
        'wall_time_s': metrics.get('wall_time_s'),
    }, ensure_ascii=False, indent=2))

In [ ]:
results = {}
for name, template in (
    ('ge_fused_o_proj_off', BASELINE_TEMPLATE),
    ('ge_fused_o_proj_16m', PREFETCH_16M_TEMPLATE),
    ('ge_fused_o_proj_32m', PREFETCH_32M_TEMPLATE),
):
    print(f'\n===== {name} =====', flush=True)
    results[name] = run_case(name, template)
    show(results[name])

In [ ]:
# 初始化日志用于确认服务器实际加载了修改后的 modeling_qwen.py，
# 而不只是Notebook读取到了正确YAML。
for label, expected_size in (
    ('ge_fused_o_proj_16m', 16 * 1024**2),
    ('ge_fused_o_proj_32m', 32 * 1024**2),
):
    log_path = Path(results[label]['recipe_log_path'])
    log_text = log_path.read_text(encoding='utf-8', errors='replace')
    expected = (
        f'Qwen weight prefetch enabled: target=self_attn.o_proj, '
        f'size={expected_size} bytes, phase=decode, dependency=value_cache_after_scatter'
    )
    if expected not in log_text:
        raise RuntimeError(
            f'{label}未找到o_proj Prefetch初始化日志：{expected}；日志={log_path}'
        )
    print(f'{label}: 初始化日志已确认，size={expected_size} bytes')

## 4 汇总性能与输出一致性

一次运行只能验证功能和趋势。正式结论至少交替运行5轮，使用中位数；Profiler开启时的数据只用于解释原因。

In [ ]:
def generated_texts(metrics):
    return [
        json.loads(line).get('generated_text', '')
        for line in Path(metrics['outputs_path']).read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

def gain(new, old):
    return (new / old - 1) * 100 if new is not None and old else None

def reduction(new, old):
    return (1 - new / old) * 100 if new is not None and old else None

baseline = results['ge_fused_o_proj_off']
baseline_texts = generated_texts(baseline)
summary = []
for label in ('ge_fused_o_proj_16m', 'ge_fused_o_proj_32m'):
    item = results[label]
    texts = generated_texts(item)
    similarity = min(
        (SequenceMatcher(None, left, right).ratio() for left, right in zip(baseline_texts, texts)),
        default=0.0,
    )
    summary.append({
        'case': label,
        'target': 'self_attn.o_proj',
        'prefetch_size_bytes': item['prefetch_status']['max_size_bytes'],
        'tokens_per_second': item.get('tokens_per_second'),
        'throughput_gain_pct': gain(item.get('tokens_per_second'), baseline.get('tokens_per_second')),
        'decode_time_s': item.get('decode_time_s'),
        'decode_time_reduction_pct': reduction(item.get('decode_time_s'), baseline.get('decode_time_s')),
        'output_similarity_ratio': similarity,
    })
output = {
    'baseline_tokens_per_second': baseline.get('tokens_per_second'),
    'baseline_decode_time_s': baseline.get('decode_time_s'),
    'comparisons': summary,
}
comparison_path = OUTPUT_ROOT / 'ge_fused_o_proj_prefetch_comparison.json'
comparison_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(output, ensure_ascii=False, indent=2))
print('结果文件 =', comparison_path)

## 5 如何确认机制有效

另外复制三份YAML并设置`enable_profiler: True`。先用PipeUtilization Profile确认每个Decode Step出现36次CMO/Prefetch，且Prefetch位于Fused Attention和`o_proj` MatMul之前；比较Fused Attention是否劣化、`o_proj`节省多少以及Free/Wait是否增加。再将Profiler指标改成L2Cache，比较`o_proj`的L2读命中率。

判断净收益：

```text
o_proj节省时间 - Fused Attention劣化时间 - 新增调度/等待 > 0
```

16 MiB约覆盖完整BF16 `o_proj`的50%，32 MiB完整覆盖o_proj权重。以`v_cache_after_scatter`作为依赖点后，CMO与Fused Attention并行执行，在FA读KV Cache期间把o_proj权重搬入L2；FA本身带宽敏感，需重点观察其是否劣化。比较两组的L2命中率与Fused Attention劣化，即可判断预取量从50%提升到完整覆盖是否仍有净收益。